# Data Pipeline for Senate LDA Input Filing Data
## aka, What Lobbying firms are people working with and on what issues?
---



## Input Requirements
- To run this pipeline you will need to first get a csv from lobbyfinder.py
- This pipeline is designed to use all of the inbuild column names from lobbyfinder.py so I don't think it will work with anything else

## To Use
- Make sure all requirements are installed, as I installed the ones I needed as I went
- import your .csv into this folder and change the pdf.readcsv to your .csv name

## Output
- This will leave you with 4 graphics in the Viz folder all built with ploty so that you can interact with them in the notebook to check values and dates. 

### Libraries Used
- pandas, plotly, kaledio, nbformat

### DataFrames in this Pipeline

---
### Last Update: 04/26/26
---

## Step 1: Dependencies and Importing Data

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

import sys
!{sys.executable} -m pip install kaleido

import plotly.io as pio
pio.kaleido.scope.mathjax = None


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
C:\Users\craig\AppData\Local\Temp\ipykernel_27348\2036799430.py:9: DeprecationWarning: 
Use of plotly.io.kaleido.scope.mathjax is deprecated and support will be removed after September 2025.
Please use plotly.io.defaults.mathjax instead.

  pio.kaleido.scope.mathjax = None


In [2]:
df = pd.read_csv('lobbying_filings_Primes.csv')
df.head()

,source,query_company,role,filing_uuid,filing_year,period,filing_type,dt_posted,registrant_name,registrant_id,client_name,client_id,income,expenses,lobbying_issues,url
0,Senate LDA,Lockheed,client,49117f1a-a236-4583-a60e-919cc59e79c4,1999,NaN,Year-End Report,2000-02-14T00:00:00-05:00,THE RHOADS GROUP,6644,LOCKHEED MARTIN CORP,108908,120000.0,NaN,Defense; Telecommunications,https://lda.senate.gov/filings/public/filing/4...
1,Senate LDA,Lockheed,client,31f28e72-7236-4837-b0b6-8d262d1d6cd8,1999,NaN,Year-End Report,2000-02-14T00:00:00-05:00,VAN SCOYOC ASSOCIATES,39837,LOCKHEED MARTIN MISSILES & SPACE CO,147700,20000.0,NaN,Defense; Budget/Appropriations,https://lda.senate.gov/filings/public/filing/3...
2,Senate LDA,Lockheed,client,5d60eeb6-241e-4977-b0d1-852a36436cbf,1999,NaN,Mid-Year Termination (No Activity),1999-08-13T00:00:00-04:00,"CARTER GROUP, LLC",49008,LOCKHEED MARTIN AIRCRAFT & LOGISTICS CENTERS,156072,NaN,NaN,NaN,https://lda.senate.gov/filings/public/filing/5...
3,Senate LDA,Lockheed,client,d52d60e0-0536-4207-8d87-9c1084e31025,1999,NaN,Year-End Report,2000-02-14T00:00:00-05:00,O'MELVENY & MYERS LLP,29904,LOCKHEED MARTIN CORP,135265,100000.0,NaN,Aerospace,https://lda.senate.gov/filings/public/filing/d...
4,Senate LDA,Lockheed,client,4e6ca9fd-7bad-451e-8a13-d7a698af4815,1999,NaN,Year-End Report,2000-02-14T00:00:00-05:00,DAP & ASSOC,48986,LOCKHEED MARTIN CORP,156067,120000.0,NaN,Telecommunications; Communications/Broadcastin...,https://lda.senate.gov/filings/public/filing/4...


## Step 2 - Data Cleaning

In [3]:
# Filter for [role] = client

df_clean = df[df['role'] == 'client']


In [15]:
# Finding each Lobbying firm by Filing year

df_clean.groupby(['registrant_name', 'filing_year'])['income'].sum().reset_index().head(10).style.format({'income': '${:,.0f}'})


,registrant_name,filing_year,income
0,"535 GROUP, LLC",2021,"$50,000"
1,"535 GROUP, LLC",2022,"$50,000"
2,"535 GROUP, LLC",2023,"$50,000"
3,"535 GROUP, LLC",2024,"$50,000"
4,"535 GROUP, LLC",2025,"$50,000"
5,"535 GROUP, LLC",2026,"$40,000"
6,AB MANAGEMENT ASSOCIATES,2003,$0
7,AB MANAGEMENT ASSOCIATES,2004,$0
8,AB MANAGEMENT ASSOCIATES,2005,$0
9,AB MANAGEMENT ASSOCIATES,2006,$0


In [14]:
# Total Income by firm per client

df_clean.groupby(['registrant_name', 'client_name'])['income'].sum().reset_index().head(10).style.format({'income': '${:,.0f}'})


,registrant_name,client_name,income
0,"535 GROUP, LLC",NORTHROP GRUMMAN SYSTEMS CORPORATION,"$290,000"
1,AB MANAGEMENT ASSOCIATES,LOCKHEED MARTIN,"$61,000"
2,ACG ADVOCACY,GENERAL DYNAMICS CORPORATION,"$530,000"
3,ACG ADVOCACY,LOCKHEED MARTIN CORP,"$50,000"
4,AKIN GUMP STRAUSS HAUER & FELD,BOEING COMPANY THE,"$2,120,000"
5,AKIN GUMP STRAUSS HAUER & FELD,RTX CORPORATION (FKA RAYTHEON TECHNOLOGIES CORPORATION AND AFFILIATES),"$1,040,000"
6,ALIGNMENT GOVERNMENT STRATEGIES,GENERAL DYNAMICS,"$400,000"
7,"ALPINE GROUP PARTNERS, LLC.",LOCKHEED MARTIN GLOBAL TELECOMMUNICATIONS,"$10,000"
8,ALSTON & BIRD LLP,BOEING DEFENSE SPACE & SECURITY (FORMERLY KESTREL ENTERPRISES INC (KEI)),"$310,000"
9,ALVARADO & BENNETT,LOCKHEED MARTIN,"$20,000"


## Step 3 - Vizualization

In [17]:
df_sankey = df_clean.groupby(['client_name', 'registrant_name'])['income'].sum().reset_index()
df_sankey = df_sankey[df_sankey['income'] > 0].dropna(subset=['income'])
df_sankey.head()


,client_name,registrant_name,income
0,"535 GROUP, LLC ON BEHALF OF NORTHROP GRUMMAN S...","RESTON STRATEGY GROUP, LLC",80000.0
1,"AEROJET ROCKETDYNE, AN L3HARRIS TECHNOLOGIES C...","FIFESTRATEGIES, LLC",420000.0
2,AVIATION PARTNERS BOEING,DENNY MILLER ASSOCIATES,165000.0
3,BAE,AMERICAN BUSINESS DEVELOPMENT GROUP,80000.0
4,BAE CORP,CONAWAY GROUP LLC,40000.0


In [18]:
clients = df_sankey['client_name'].unique().tolist()
firms = df_sankey['registrant_name'].unique().tolist()
nodes = clients + firms


In [19]:
source = [nodes.index(c) for c in df_sankey['client_name']]
target = [nodes.index(f) for f in df_sankey['registrant_name']]
value = df_sankey['income'].tolist()


In [ ]:
# Limit to top 10 clients and top 15 firms by income for readability
top_clients = df_sankey.groupby('client_name')['income'].sum().nlargest(10).index
top_firms = df_sankey.groupby('registrant_name')['income'].sum().nlargest(15).index

df_sankey_top = df_sankey[
    df_sankey['client_name'].isin(top_clients) &
    df_sankey['registrant_name'].isin(top_firms)
]

# Clients on left (source), firms on right (target) — both sorted largest to top
clients = (df_sankey_top.groupby('client_name')['income'].sum()
           .sort_values(ascending=False).index.tolist())
firms = (df_sankey_top.groupby('registrant_name')['income'].sum()
         .sort_values(ascending=False).index.tolist())
nodes = clients + firms

source = [nodes.index(c) for c in df_sankey_top['client_name']]
target = [nodes.index(f) for f in df_sankey_top['registrant_name']]
value = df_sankey_top['income'].tolist()

fig = go.Figure(go.Sankey(
    node=dict(label=nodes, pad=25, thickness=20, color='steelblue'),
    link=dict(source=source, target=target, value=value, color='rgba(100,149,237,0.3)')
))
fig.update_layout(
    title='Top Lobbying Firm Income by Client',
    height=700,
    font=dict(size=11),
    hoverlabel=dict(bgcolor='white', font_color='black', bordercolor='lightgrey')
)
fig.show()